## Stage 2.2 — Add Remote/In-Office Job Classification

**Role:** Second notebook of Stage 2. Classifies each job posting as `remote`, `in_office`, `combined`, or `undefined` using a combination of multilingual regex hints and semantic similarity against handcrafted templates in five languages (EN, UA, RU, PL, CZ).

**Position:** Run after `stage_2_1_skills_extration.ipynb`. Updates the same output pickles produced by Stage 2.1 in-place by adding `job_type` and `job_type_score` columns.

**Output:** Stage 2.1 output pickles updated with work-mode classification columns.

**Documentation:** [Notebook guide](README.md) · [Stage data description](../../data/data-pipeline/stage_02/README_data.md)


# 2. Stage 2. Candidate Skill Retrieval

It reads **`stage1\output\***.pkl`** (output of the previous step), embeds each posting’s description with the appropriate *sentence‑transformer* model, retrieves the top‑K most similar ESCO **skill IDs**, and writes to **`stage2\output\****.pkl`**

## 2.1. Load packages and configuration

Load autoreload and import shared modules. `pipeline` from `transformers` is imported for potential NLP use. Call `clean_memory()` to release variables from previous runs.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pipeline_bootstrap import configure_pipeline
configure_pipeline()
import general as g
import os
g.clean_memory()
import pandas as pd
import stage1 as st1
from transformers import pipeline

In [ ]:
# Fast multilingual (UA/EN/RU/CZ/PL) detection of job work mode:
# remote | in_office | combined | undefined
#
# Approach:
# - quick regex hints per language
# - semantic similarity to multilingual templates via sentence-transformers
#
# Requirements:
#   pip install -U sentence-transformers regex
#
# Model choice: small & fast multilingual model
#   "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
#   (works for UA/EN/RU/CZ/PL and runs well on CPU; GPU if available)

import re
import math
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass

import torch
from sentence_transformers import SentenceTransformer, util

Define the work-mode detection system.

**Model:** `paraphrase-multilingual-MiniLM-L12-v2` — small, fast, and effective on CPU.

**Templates:** Manually authored reference sentences for three work modes (`remote`, `in_office`, `combined`) in all five pipeline languages. These are encoded once at load time and cached as `_template_emb`.

**Regex hints:** Fast pre-filters in all languages that nudge the classifier before running the semantic model, improving speed and accuracy on clear-cut cases.

**`detect_work_mode()`:** Core classification function. Splits the description into chunks, encodes each chunk, computes cosine similarity against all templates, aggregates per-class scores, applies hint nudges, and returns a `Detection` object with a label, per-class scores, and fired hints.

In [ ]:
MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------------------
# Templates in UA/EN/RU/PL/CZ
# ---------------------------
TEMPLATES = {
    "remote": [
        # EN
        "This job is remote.",
        "Remote work is allowed.",
        "Work from home.",
        # UA
        "Ця робота віддалена.",
        "Дозволена віддалена робота.",
        "Робота з дому.",
        # RU
        "Эта работа удалённая.",
        "Разрешена удалённая работа.",
        # PL
        "Ta praca jest zdalna.",
        "Dozwolona praca zdalna.",
        # CZ
        "Tato práce je na dálku.",
        "Je povolena práce na dálku.",
    ],
    "in_office": [
        # EN
        "This job is in the office only.",
        "On-site work is required.",
        # UA
        "Робота лише в офісі.",
        "Потрібна робота в офісі.",
        # RU
        "Работа только в офисе.",
        "Требуется работа в офисе.",
        # PL
        "Praca wyłącznie stacjonarna.",
        "Wymagana praca w biurze.",
        # CZ
        "Práce pouze v kanceláři.",
        "Je vyžadována práce v kanceláři.",
    ],
    "combined": [
        # EN
        "This job is hybrid (remote and in-office).",
        "The position combines remote and office work.",
        # UA
        "Ця робота гібридна (віддалено та в офісі).",
        "Посада поєднує віддалену та офісну роботу.",
        # RU
        "Эта работа гибридная (удалённо и в офисе).",
        "Должность сочетает удалённую и офисную работу.",
        # PL
        "Ta praca jest hybrydowa (zdalnie i w biurze).",
        "Stanowisko łączy pracę zdalną i biurową.",
        # CZ
        "Tato práce je hybridní (na dálku i v kanceláři).",
        "Pozice kombinuje práci na dálku a v kanceláři.",
    ],
}

# ---------------------------
# Very fast hint regex (multi-lang)
# ---------------------------
FLAG = re.IGNORECASE | re.UNICODE

REMOTE_HINT = re.compile(
    r"(remote|wfh|work\s*from\s*home|віддален\w+|дистанційн\w+|удалённ\w+|удаленн\w+|"
    r"дистанционн\w+|zdaln\w+|na\s*dálku)", FLAG)

OFFICE_HINT = re.compile(
    r"(on[-\s]*site|in[-\s]*office|office\s+only|лише\s*офіс|тільки\s*офіс|"
    r"в\s*офіс(е|і)|только\s*в\s*офисе|stacjonarn\w+|w\s*biurze|"
    r"pouze\s*v\s*kanceláři|v\s*kanceláři)", FLAG)

HYBRID_HINT = re.compile(
    r"(hybrid|гібрид\w+|гибрид\w+|hybrydow\w+|hybridn\w+)", FLAG)

NO_REMOTE_NEGATION = re.compile(
    r"(no\s+remote|без\s*віддал|без\s*удал|brak\s+pracy\s+zdalnej|"
    r"tylko\s*w\s*biurze|only\s*(on[-\s]*site|in[-\s]*office))", FLAG)

# Avoid tech false positives like "remote sensing/desktop"
TECH_REMOTE_FALSE = re.compile(
    r"\bremote\s+(sensing|control|desktop|server|repository|branch|machine)\b", FLAG
)

# ---------------------------
# Simple sentence splitter (fast)
# ---------------------------
SENT_SPLIT = re.compile(r"(?<=[\.\!\?…])\s+|\n{2,}")

@dataclass
class Detection:
    label: str  # "remote" | "in_office" | "combined" | "undefined"
    scores: Dict[str, float]  # cosine scores by class
    hints: Dict[str, bool]    # which hints fired

# ---------------------------
# Model (load once)
# ---------------------------
_model = SentenceTransformer(MODEL_ID, device=DEVICE)
# Pre-encode templates (speed!)
_template_texts: List[str] = []
_template_index: List[Tuple[str, int]] = []  # (class, idx)
for cls, arr in TEMPLATES.items():
    for i, t in enumerate(arr):
        _template_texts.append(t)
        _template_index.append((cls, i))
_template_emb = _model.encode(_template_texts, convert_to_tensor=True, normalize_embeddings=True)

# ---------------------------
# Core functions
# ---------------------------
def _chunks(text: str, max_chars: int = 800) -> List[str]:
    """Fast chunker: split by sentences/paragraphs, cap length."""
    parts = SENT_SPLIT.split(text.strip())
    chunks, buf = [], ""
    for p in parts:
        p = p.strip()
        if not p:
            continue
        cand = (buf + " " + p).strip() if buf else p
        if len(cand) <= max_chars:
            buf = cand
        else:
            if buf:
                chunks.append(buf)
            # hard cut long part
            for i in range(0, len(p), max_chars):
                chunks.append(p[i:i+max_chars])
            buf = ""
    if buf:
        chunks.append(buf)
    return chunks if chunks else [text]

def _semantic_scores(text: str) -> Dict[str, float]:
    """Compute per-class similarity: max over (chunks × templates of that class), then mean across chunks."""
    chunks = _chunks(text)
    ch_emb = _model.encode(chunks, convert_to_tensor=True, normalize_embeddings=True)
    # cosine sim for all chunks vs all templates (batched and fast)
    sims = util.cos_sim(ch_emb, _template_emb)  # [n_chunks, n_templates]
    # reduce: for each class, take max over its templates per chunk, then mean over chunks
    per_class_scores: Dict[str, List[float]] = {k: [] for k in TEMPLATES.keys()}
    for ci in range(sims.size(0)):  # each chunk
        row = sims[ci]  # [n_templates]
        # gather class-wise best
        for cls in TEMPLATES.keys():
            # column indices for this class
            idxs = [j for j, (cname, _) in enumerate(_template_index) if cname == cls]
            best = float(row[idxs].max().item())
            per_class_scores[cls].append(best)
    # aggregate
    return {cls: (sum(v)/len(v) if v else 0.0) for cls, v in per_class_scores.items()}

def detect_work_mode(description: str, title: Optional[str] = None) -> Detection:
    """
    Returns Detection(label, scores, hints)
    Labels: 'remote' | 'in_office' | 'combined' | 'undefined'
    """
    text = ((title or "").strip() + "\n" + (description or "").strip()).strip()
    if not text:
        return Detection("undefined", {}, {"remote": False, "office": False, "hybrid": False, "no_remote": False})

    # Quick guard for "remote sensing/desktop" etc.
    if TECH_REMOTE_FALSE.search(text):
        return Detection("undefined", {"remote": 0.0, "in_office": 0.0, "combined": 0.0},
                         {"remote": False, "office": False, "hybrid": False, "no_remote": False})

    # Hints (cheap)
    hint_remote = bool(REMOTE_HINT.search(text))
    hint_office = bool(OFFICE_HINT.search(text))
    hint_hybrid = bool(HYBRID_HINT.search(text))
    hint_no_remote = bool(NO_REMOTE_NEGATION.search(text))

    # Semantic scores (fast encoder)
    scores = _semantic_scores(text)  # remote / in_office / combined

    # Small nudges based on hints (very light, keeps semantics primary)
    if hint_no_remote:
        scores["in_office"] += 0.03
    if hint_hybrid:
        scores["combined"] += 0.03
    if hint_remote and not hint_office:
        scores["remote"] += 0.01
    if hint_office and not hint_remote:
        scores["in_office"] += 0.01

    # Decision rules
    r, o, c = scores["remote"], scores["in_office"], scores["combined"]

    # If very weak evidence across the board
    if max(r, o, c) < 0.28:  # default threshold for this model; adjust if needed
        return Detection("undefined", scores, {
            "remote": hint_remote, "office": hint_office, "hybrid": hint_hybrid, "no_remote": hint_no_remote
        })

    # Combined if hybrid wins or both remote & office are strong and close
    delta = 0.03
    if c >= r and c >= o:
        return Detection("combined", scores, {
            "remote": hint_remote, "office": hint_office, "hybrid": hint_hybrid, "no_remote": hint_no_remote
        })
    if r >= 0.32 and o >= 0.32 and abs(r - o) <= 0.05:
        return Detection("combined", scores, {
            "remote": hint_remote, "office": hint_office, "hybrid": hint_hybrid, "no_remote": hint_no_remote
        })

    # Otherwise pick stronger of remote vs in_office
    label = "remote" if r > o else "in_office"
    return Detection(label, scores, {
        "remote": hint_remote, "office": hint_office, "hybrid": hint_hybrid, "no_remote": hint_no_remote
    })

# ---------------------------
# Batch helper (list of dicts or pandas.DataFrame)
# ---------------------------
def detect_batch(records, text_col="description", title_col="title"):
    try:
        import pandas as pd
        is_df = isinstance(records, pd.DataFrame)
    except Exception:
        is_df = False

    def _one(rec):
        if isinstance(rec, dict):
            title, desc = rec.get(title_col, ""), rec.get(text_col, "")
        else:
            title, desc = getattr(rec, title_col, ""), getattr(rec, text_col, "")
        d = detect_work_mode(desc or "", title or "")
        out = {"work_mode": d.label, "scores": d.scores, "hints": d.hints}
        return ({**rec, **out} if isinstance(rec, dict) else out)

    if is_df:
        import pandas as pd
        return records.apply(lambda row: _one(row.to_dict()), axis=1, result_type="expand")
    else:
        return [_one(r) for r in records]

Define helper functions for robust score extraction and batch classification.

`_extract_label_and_score()` handles multiple possible result shapes (object, dict, tuple/list) to safely extract the label and confidence score regardless of how `detect_work_mode()` returns its result.

`add_job_type_by_max_score()` applies work-mode detection to an entire DataFrame, writing the predicted label to `job_type` and its score to `job_type_score`.

In [ ]:
from typing import Any, Tuple, Optional

def _extract_label_and_score(det: Any) -> Tuple[Optional[str], Optional[float]]:
    """
    Try to extract (label, score) from various detection result shapes:
    - Object with .label and one of .score/.confidence/.max_score/.prob/.probability
    - Object with .scores (dict/list/tuple) -> take max for score
    - Dict with keys 'label'/'type'/'name' and 'score'/'confidence'/'max_score'/'prob'
    - Tuple/list like (label, score, ...)
    """
    if det is None:
        return None, None

    # Tuple or list form
    if isinstance(det, (tuple, list)):
        label = det[0] if len(det) >= 1 else None
        score = det[1] if len(det) >= 2 else None
        return label, score

    # Dict form
    if isinstance(det, dict):
        label = det.get("label") or det.get("type") or det.get("name")
        score = det.get("score") or det.get("confidence") or det.get("max_score") or det.get("prob") or det.get("probability")
        if score is None:
            scores = det.get("scores")
            if isinstance(scores, dict) and scores:
                score = max(scores.values())
            elif isinstance(scores, (list, tuple)) and scores:
                score = max(scores)
        return label, score

    # Object form
    label = getattr(det, "label", None) or getattr(det, "type", None) or getattr(det, "name", None)

    score = None
    for attr in ("score", "confidence", "max_score", "prob", "probability"):
        val = getattr(det, attr, None)
        if val is not None:
            score = val
            break

    if score is None:
        scores = getattr(det, "scores", None)
        if isinstance(scores, dict) and scores:
            score = max(scores.values())
        elif isinstance(scores, (list, tuple)) and scores:
            score = max(scores)

    return label, score


def add_job_type_by_max_score(
    df,
    title_col="clean_title",
    desc_col="clean_desc",
    out_label_col="job_type",
    out_score_col="job_type_score",
    round_score=4,
):
    """
    Adds job type classification and its score to the dataframe using detect_work_mode.
    This version robustly extracts the score from various Detection result shapes
    and avoids assuming a .score attribute exists.
    """
    labels = []
    scores_out = []

    for _, row in df.iterrows():
        title = row.get(title_col)
        desc = row.get(desc_col)

        det = detect_work_mode(str(desc or ""), str(title or ""))

        label, score = _extract_label_and_score(det)
        labels.append(label)

        if score is not None and round_score is not None:
            try:
                score = round(float(score), round_score)
            except (TypeError, ValueError):
                # If score is non-numeric, keep as-is (or set to None)
                score = None

        scores_out.append(score)

    df[out_label_col] = labels
    df[out_score_col] = scores_out
    return df

Load the Stage 2 process tracker and add a `remote_status` column if not present. This column tracks which files have had work-mode classification applied.

In [ ]:
proc_df = pd.read_pickle(g.Config.STAGE2_PROCESS_PATH)

if "remote_status" not in proc_df.columns:
    proc_df["remote_status"] = None
proc_df

Import datetime utilities for timing each file's classification step.

In [ ]:
from datetime import datetime, timedelta

Load the Stage 2 process tracker and add a `remote_status` column if not present. This column tracks which files have had work-mode classification applied.

In [ ]:
for _,row in proc_df.iterrows():
    if row["remote_status"] != "complete":
        start_dt = datetime.now()
        data = pd.read_pickle(row["extract_path"]).copy()
        data = add_job_type_by_max_score(data, title_col="clean_title", desc_col="clean_desc")
        proc_df.loc[_, "remote_status"] = "complete"
        data.to_pickle(row["extract_path"])
        end_dt = datetime.now()
        print(f"- {row['input_file']} ({data.shape[0]}) in {end_dt - start_dt}")
    proc_df.to_pickle(g.Config.STAGE2_PROCESS_PATH)

Verification cell. Load the output pickle for the demo file and inspect the last rows to confirm `job_type` and `job_type_score` columns are populated.

In [ ]:
dt = pd.read_pickle(os.path.join(g.Config.STAGE2_OUTPUT_PATH, "ua-2024-01-01.pkl"))
dt.tail()

---
© 2026 Yurii Kleban, Britta Rude. All rights reserved.